# Lab 4 · Capstone: August breaks the model
**Block 2 · about 30 min.** A competitor launched in July and the BASIC plan got more expensive.
You get the August data with labels. Your job: **detect, retrain, gate, promote** — all tracked in MLflow.

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")          # churn_data.py, churn_model.py
warnings.filterwarnings("ignore")

import mlflow
mlflow.set_tracking_uri("http://127.0.0.1:5000")   # the server you started in the terminal
mlflow.set_experiment("churn-workshop")
print("MLflow", mlflow.__version__)

In [ ]:
import numpy as np, pandas as pd
from mlflow import MlflowClient
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from churn_data import history, august, FEATURES, NUMERIC, TARGET
from churn_model import make_pipeline, XGB_TRUSTED_TYPES

client, NAME = MlflowClient(), "churn-model"
hist, aug = history(), august()
# Native sklearn flavor (not pyfunc) so we get probabilities with predict_proba
champion = mlflow.sklearn.load_model(f"models:/{NAME}@champion")
score = lambda model, X: model.predict_proba(X)[:, 1]
champ_version = client.get_model_version_by_alias(NAME, "champion").version
print("current champion: version", champ_version)

## Step 1 · Detect drift (PSI per feature)
Population Stability Index: < 0.1 stable · 0.1–0.25 investigate · > 0.25 act.

In [ ]:
def psi(expected, actual, bins=10):
    edges = np.unique(np.quantile(expected, np.linspace(0, 1, bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    e = np.histogram(expected, edges)[0] / len(expected) + 1e-6
    a = np.histogram(actual, edges)[0] / len(actual) + 1e-6
    return float(np.sum((a - e) * np.log(a / e)))

baseline_df = hist[hist.SNAPSHOT_TS >= "2026-05-01"]
drift = {f: round(psi(baseline_df[f], aug[f]), 3) for f in NUMERIC}
drift

## Step 2 · Confirm with labels: how does the champion do on August?

In [ ]:
auc_jun = roc_auc_score(baseline_df[TARGET], score(champion, baseline_df[FEATURES]))
auc_aug = roc_auc_score(aug[TARGET], score(champion, aug[FEATURES]))
print(f"champion AUC  May-Jun: {auc_jun:.3f}   August: {auc_aug:.3f}")

## Step 3 · Retrain a challenger and gate it, in one tracked run
Both models are scored on the **same fresh holdout** (30% of August, never used for training).

In [ ]:
holdout = aug.sample(frac=0.3, random_state=7)
train_new = pd.concat([hist, aug.drop(holdout.index)])
MARGIN = 0.01

with mlflow.start_run(run_name="retrain_2026_08") as run:
    mlflow.log_dict(drift, "drift/psi.json")
    mlflow.log_metrics({"psi_logins": drift["LOGINS_30D"], "champion_auc_aug": auc_aug})
    mlflow.log_input(mlflow.data.from_pandas(train_new, name="churn_history_plus_aug",
                                             targets=TARGET), context="training")

    challenger = make_pipeline(XGBClassifier(max_depth=4, n_estimators=300, learning_rate=0.05,
                                             eval_metric="logloss"))
    challenger.fit(train_new[FEATURES], train_new[TARGET])

    auc_champ = roc_auc_score(holdout[TARGET], score(champion, holdout[FEATURES]))
    auc_chal = roc_auc_score(holdout[TARGET], score(challenger, holdout[FEATURES]))
    passed = auc_chal > auc_champ + MARGIN
    mlflow.log_metrics({"holdout_auc_champion": auc_champ, "holdout_auc_challenger": auc_chal})
    mlflow.set_tag("gate", "passed" if passed else "failed")

    info = mlflow.sklearn.log_model(challenger, name="churn_xgb_retrained",
                                    input_example=holdout[FEATURES].head(3),
                                    skops_trusted_types=XGB_TRUSTED_TYPES,
                                    registered_model_name=NAME)

print(f"champion {auc_champ:.3f} vs challenger {auc_chal:.3f} -> gate {'PASSED' if passed else 'FAILED'}")

## Step 4 · Promote — or not
The new version is registered either way (evidence), but only a winner gets `@champion`.
The old champion keeps an alias `@previous` so rollback is one call.

In [ ]:
new_version = info.registered_model_version
client.set_model_version_tag(NAME, new_version, "gate", "passed" if passed else "failed")
client.set_registered_model_alias(NAME, "challenger", new_version)
if passed:
    client.set_registered_model_alias(NAME, "previous", champ_version)
    client.set_registered_model_alias(NAME, "champion", new_version)

for alias in ["champion", "challenger", "previous", "baseline"]:
    try:
        print(f"@{alias:<10} -> version {client.get_model_version_by_alias(NAME, alias).version}")
    except Exception:
        print(f"@{alias:<10} -> (not set)")

👉 **In the UI:** the `retrain_2026_08` run tells the whole story: drift numbers, data used, both AUCs, the gate decision
and the registered version. That is your audit trail.

## ✍️ Your turn — discussion (5 min)
1. The challenger won. Is retraining enough, or does the BASIC price change call for a **new feature**?
2. Rollback drill: point `@champion` back to `@previous` in one line.
3. Which parts of this notebook would you move into a scheduled job? Which should stay a human decision?